In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ----------------------------------------------------------------------
# 1. Setup and Imports (UPDATED FOR DENSENET201 AND ADVANCED LIBRARIES)
# ----------------------------------------------------------------------
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import DenseNet201 # NEW: Using DenseNet201
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils import class_weight # NEW: For class weighting
import tensorflow as tf
# Use DenseNet201 specific preprocessing
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess_input 

# Grad-CAM specific imports
import cv2
from tensorflow.keras import backend as K

# Define base directory
base_dir = '/kaggle/input/pneumonia-detection-dataset/chest_xray/'
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')

# Define class labels for the multi-class task (3 CLASSES)
CLASS_LABELS = ['NORMAL', 'PNEUMONIA_BACTERIA', 'PNEUMONIA_VIRUS']
NUM_CLASSES = len(CLASS_LABELS)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# ----------------------------------------------------------------------
# 2. Data Preparation and Pre-processing (CRITICAL: CUSTOM DATA GENERATION)
# ----------------------------------------------------------------------
def create_3_class_df(data_dir):
    """
    Parses filenames to create a DataFrame with explicit 3-class labels.
    """
    filepaths = []
    labels = []
    
    # Process NORMAL images
    normal_path = os.path.join(data_dir, 'NORMAL')
    if os.path.exists(normal_path):
        for filename in os.listdir(normal_path):
            filepaths.append(os.path.join(normal_path, filename))
            labels.append('NORMAL')
            
    # Process PNEUMONIA images
    pneumonia_path = os.path.join(data_dir, 'PNEUMONIA')
    if os.path.exists(pneumonia_path):
        for filename in os.listdir(pneumonia_path):
            filepath = os.path.join(pneumonia_path, filename)
            filepaths.append(filepath)
            
            if 'bacteria' in filename.lower():
                labels.append('PNEUMONIA_BACTERIA')
            elif 'virus' in filename.lower():
                labels.append('PNEUMONIA_VIRUS')
            else:
                # Assign to the majority class if type is unknown (standard behavior)
                labels.append('PNEUMONIA_BACTERIA') 

    df = pd.DataFrame({'filepaths': filepaths, 'labels': labels})
    df = df[df['labels'].isin(CLASS_LABELS)] 
    
    return df

# Create DataFrames for Training and Testing
train_df = create_3_class_df(train_dir)
test_df = create_3_class_df(test_dir)

print(f"Train set distribution:\n{train_df['labels'].value_counts()}")
print(f"Test/Validation set distribution:\n{test_df['labels'].value_counts()}")

# --- NEW: Calculate Class Weights for Imbalance Handling ---
class_indices = {label: i for i, label in enumerate(CLASS_LABELS)}
true_labels_int = train_df['labels'].map(class_indices).values

class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(true_labels_int),
    y=true_labels_int
)
class_weight_dict = dict(enumerate(class_weights))
print(f"\nCalculated Class Weights for training: {class_weight_dict}")
# -----------------------------------------------------------

# ----------------------------------------------------------------------
# 3. Data Augmentation and Generators (UPDATED: DENSENET201 & BRIGHTNESS)
# ----------------------------------------------------------------------

# Training Data Generator (Aggressive augmentation + Contrast control)
train_datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2], # NEW: Added brightness augmentation
    fill_mode='nearest',
    preprocessing_function=densenet_preprocess_input 
)

# Test/Validation Data Generator (Only resizing and preprocessing)
test_val_datagen = ImageDataGenerator(
    preprocessing_function=densenet_preprocess_input
)

train_generator = train_datagen.flow_from_dataframe(
    train_df,
    x_col='filepaths',
    y_col='labels',
    target_size=IMG_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_generator = test_val_datagen.flow_from_dataframe(
    test_df,
    x_col='filepaths',
    y_col='labels',
    target_size=IMG_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ----------------------------------------------------------------------
# 4. Model Architecture (UPDATED: DENSENET201 TRANSFER LEARNING)
# ----------------------------------------------------------------------

# Load the DenseNet201 model pre-trained on ImageNet
base_model = DenseNet201(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the convolutional base
base_model.trainable = False

# Create the new classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ----------------------------------------------------------------------
# 5. Training (Fine-Tuning Strategy)
# ----------------------------------------------------------------------

# Phase 1: Train only the top layers (new head)
print("\n--- Phase 1: Training Classification Head with Class Weights ---")
history_head = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=15,
    validation_data=test_generator,
    validation_steps=test_generator.samples // BATCH_SIZE,
    class_weight=class_weight_dict # CRITICAL: Apply class weights here
)

# Phase 2: Fine-tune the entire network (unfreeze top blocks)
base_model.trainable = True

# NEW: Only unfreeze the last 50 layers for more stable fine-tuning of subtle features
for layer in base_model.layers[:-50]: 
    layer.trainable = False

# Re-compile the model with a much lower learning rate for fine-tuning
model.compile(
    optimizer=Adam(learning_rate=1e-5), # Very low learning rate
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- Phase 2: Fine-Tuning Top 50 Layers with Class Weights ---")
history_full = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=10, # Reduced epochs for fine-tuning stability
    validation_data=test_generator,
    validation_steps=test_generator.samples // BATCH_SIZE,
    class_weight=class_weight_dict # CRITICAL: Apply class weights here
)

# ----------------------------------------------------------------------
# 6. Evaluation (UPDATED: MULTI-CLASS METRICS)
# ----------------------------------------------------------------------

# Generate predictions
test_generator.reset()
y_pred_probs = model.predict(test_generator, steps=test_generator.samples // BATCH_SIZE + 1)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

# Get true labels and class names
true_classes = test_generator.classes
class_indices = test_generator.class_indices
class_labels = list(class_indices.keys())

# Ensure y_pred_classes and true_classes have the same length
if len(y_pred_classes) < len(true_classes):
    true_classes = true_classes[:len(y_pred_classes)]
else:
    y_pred_classes = y_pred_classes[:len(true_classes)]

# Classification Report
print("\n--- Classification Report (Multi-Class) ---")
print(classification_report(true_classes, y_pred_classes, target_names=class_labels))

# Confusion Matrix
cm = confusion_matrix(true_classes, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title('Multi-Class Confusion Matrix (DenseNet201 + Class Weights)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


# --- Visualization of Training History ---
def plot_training_history(history, title):
    """Plots the training and validation loss and accuracy."""
    plt.figure(figsize=(12, 4))

    # Plot Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()
    
    
print("\n--- Training History Plots ---")
plot_training_history(history_head, "Phase 1: Classification Head Training")
plot_training_history(history_full, "Phase 2: Top Layers Fine-Tuning")


# ----------------------------------------------------------------------
# 7. Prediction and Grad-CAM Interpretability (SAME LOGIC, NEW MODEL)
# ----------------------------------------------------------------------

def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    """Generates Grad-CAM heatmap for a given image and model."""
    
    # 1. Create a model that maps the input image to the activations of the last conv layer as well as the final predictions
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    # 2. Get the last convolutional layer output and the model prediction
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        
        # Get the predicted class index
        pred_index = tf.argmax(preds[0])
        
        # Get the loss for the predicted class
        class_channel = preds[:, pred_index]

    # 3. Compute the gradient of the predicted class output with respect to the output of the last conv layer
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # 4. Pool the gradients to obtain the weights
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # 5. Compute the heatmap
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # 6. Normalize the heatmap
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def display_gradcam(img_path, model, target_size, last_conv_layer_name="conv5_block32_concat"): 
    """Displays the original image and the Grad-CAM heatmap overlay.
       Using 'conv5_block32_concat' which is a common last layer in DenseNet201."""
    
    # Load and preprocess the image
    img = load_img(img_path, target_size=target_size)
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    # Use DenseNet preprocessing
    img_array_preprocessed = densenet_preprocess_input(img_array.copy()) 

    # Get the predicted class and label
    predictions = model.predict(img_array_preprocessed)
    pred_index = np.argmax(predictions[0])
    predicted_label = class_labels[pred_index]
    
    # Generate heatmap
    heatmap = make_gradcam_heatmap(img_array_preprocessed, model, last_conv_layer_name=last_conv_layer_name)

    # Rescale heatmap to be same size as original image
    heatmap = cv2.resize(heatmap, (img_array.shape[2], img_array.shape[1]))
    heatmap = 255 * heatmap
    heatmap = np.uint8(heatmap)
    
    # Apply colormap (Jet is common for heatmaps)
    jet = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    
    # Superimpose the heatmap on the original image
    superimposed_img = jet * 0.4 + img_array[0]
    superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)

    # Display results
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(img_array[0].astype('uint8'))
    ax[0].set_title(f"Original Image (Predicted: {predicted_label})")
    ax[0].axis('off')
    
    ax[1].imshow(superimposed_img)
    ax[1].set_title(f"Grad-CAM Visualization (Focus)")
    ax[1].axis('off')
    
    plt.suptitle(f"Prediction: {predicted_label} | Probabilities: {predictions[0]}")
    plt.show()


# Example Usage of Prediction and Grad-CAM (using a sample file from the test set)
sample_file_bacteria = '/kaggle/input/pneumonia-detection-dataset/chest_xray/test/PNEUMONIA/person103_bacteria_490.jpeg'
sample_file_virus = '/kaggle/input/pneumonia-detection-dataset/chest_xray/test/PNEUMONIA/person1614_virus_2800.jpeg'
sample_file_normal = '/kaggle/input/pneumonia-detection-dataset/chest_xray/test/NORMAL/IM-0011-0001-0001.jpeg'

print("\n--- Example Prediction: Bacterial Pneumonia with Grad-CAM ---")
# Use the known last conv layer name for DenseNet201
display_gradcam(sample_file_bacteria, model, IMG_SIZE, last_conv_layer_name='conv5_block32_concat')

print("\n--- Example Prediction: Viral Pneumonia with Grad-CAM ---")
display_gradcam(sample_file_virus, model, IMG_SIZE, last_conv_layer_name='conv5_block32_concat')

print("\n--- Example Prediction: Normal Case with Grad-CAM ---")
display_gradcam(sample_file_normal, model, IMG_SIZE, last_conv_layer_name='conv5_block32_concat')

In [ ]:
# ----------------------------------------------------------------------
# 8. Save Model Output and Artifacts
# ----------------------------------------------------------------------
import pickle
import datetime

# Define a timestamp for unique filenames
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = '/kaggle/working/'
os.makedirs(output_dir, exist_ok=True)
print(f"Saving artifacts to directory: {output_dir}")

# --- Save the Final Confusion Matrix ---
cm_filename = os.path.join(output_dir, f"confusion_matrix_{timestamp}.png")
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title('Multi-Class Confusion Matrix (DenseNet201 + Class Weights)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.savefig(cm_filename)
print(f"Saved Confusion Matrix plot to: {cm_filename}")

# --- Save Training History ---
# Combine history objects for full training visualization
full_history = {}
for key in history_head.history.keys():
    full_history[key] = history_head.history[key] + history_full.history[key]

history_filename = os.path.join(output_dir, f"training_history_{timestamp}.pkl")
with open(history_filename, 'wb') as file:
    pickle.dump(full_history, file)
print(f"Saved combined training history (head + full fine-tune) to: {history_filename}")

# --- Save Training History Plots ---
plot_history_filename_head = os.path.join(output_dir, f"history_plot_phase1_{timestamp}.png")
def save_plot_training_history(history, title, filename):
    """Saves the training and validation loss and accuracy plots."""
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(filename)
    plt.close() # Close plot to prevent display repetition

save_plot_training_history(history_head, "Phase 1: Classification Head Training", plot_history_filename_head)
print(f"Saved Phase 1 History plot to: {plot_history_filename_head}")

plot_history_filename_full = os.path.join(output_dir, f"history_plot_phase2_{timestamp}.png")
save_plot_training_history(history_full, "Phase 2: Top Layers Fine-Tuning", plot_history_filename_full)
print(f"Saved Phase 2 History plot to: {plot_history_filename_full}")


# --- Save Grad-CAM Visualizations ---
# Redefine display_gradcam to save the figure instead of showing it
def save_gradcam_visualization(img_path, model, target_size, last_conv_layer_name, output_filepath): 
    """Displays and saves the original image and the Grad-CAM heatmap overlay."""
    
    # Load and preprocess the image
    img = load_img(img_path, target_size=target_size)
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array_preprocessed = densenet_preprocess_input(img_array.copy()) 

    # Get the predicted class and label
    predictions = model.predict(img_array_preprocessed)
    pred_index = np.argmax(predictions[0])
    predicted_label = class_labels[pred_index]
    
    # Generate heatmap
    heatmap = make_gradcam_heatmap(img_array_preprocessed, model, last_conv_layer_name=last_conv_layer_name)

    # Rescale heatmap and superimpose
    heatmap = cv2.resize(heatmap, (img_array.shape[2], img_array.shape[1]))
    heatmap = 255 * heatmap
    heatmap = np.uint8(heatmap)
    jet = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    superimposed_img = jet * 0.4 + img_array[0]
    superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)

    # Display and Save results
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(img_array[0].astype('uint8'))
    ax[0].set_title(f"Original Image (Predicted: {predicted_label})")
    ax[0].axis('off')
    
    ax[1].imshow(superimposed_img)
    ax[1].set_title(f"Grad-CAM Visualization (Focus)")
    ax[1].axis('off')
    
    plt.suptitle(f"Prediction: {predicted_label} | Probabilities: {predictions[0]}")
    plt.savefig(output_filepath)
    plt.close() # Close plot after saving

# Save the examples
gradcam_bacteria_filename = os.path.join(output_dir, f"gradcam_bacteria_example_{timestamp}.png")
save_gradcam_visualization(sample_file_bacteria, model, IMG_SIZE, 'conv5_block32_concat', gradcam_bacteria_filename)
print(f"Saved Grad-CAM for Bacterial Pneumonia to: {gradcam_bacteria_filename}")

gradcam_virus_filename = os.path.join(output_dir, f"gradcam_virus_example_{timestamp}.png")
save_gradcam_visualization(sample_file_virus, model, IMG_SIZE, 'conv5_block32_concat', gradcam_virus_filename)
print(f"Saved Grad-CAM for Viral Pneumonia to: {gradcam_virus_filename}")

gradcam_normal_filename = os.path.join(output_dir, f"gradcam_normal_example_{timestamp}.png")
save_gradcam_visualization(sample_file_normal, model, IMG_SIZE, 'conv5_block32_concat', gradcam_normal_filename)
print(f"Saved Grad-CAM for Normal Case to: {gradcam_normal_filename}")

# --- Optional: Save the Final Trained Model ---
model_filename = os.path.join(output_dir, f"densenet201_3class_model_{timestamp}.keras")
model.save(model_filename)
print(f"Saved the final trained model to: {model_filename}")

print("\nAll model artifacts have been saved successfully.")